# Gaps, Islands and Sessions

## Quick notes
### What this covers
- Islands (unbroken runs) and gaps (the holes between runs) in numbers, dates and statuses.
- The `ROW_NUMBER` difference trick, consecutive-day streaks, longest streak, current streak.
- Missing IDs and missing dates (a calendar built with a recursive CTE).
- Sessionization: cut an event stream into sessions with `LAG` and a running `SUM` of "new session" flags.
- Overlapping intervals: detect, merge, measure coverage and free time.
- Status-change runs and run-length encoding.
### Key points
- Island = a maximal run of consecutive rows. Gap = a hole between two islands.
- Trick 1: for consecutive integers, `value - ROW_NUMBER()` is the same constant for the whole run. Group by it.
- Trick 2 (works for anything): flag "this row starts a new group" with `LAG`, then a running `SUM` of the flags gives a group number.
- Missing items: compare to a generated full sequence (recursive CTE) with a `LEFT JOIN` or `NOT EXISTS`.
- Two intervals overlap when `a.start < b.end AND b.start < a.end`.
### Gotchas (the part worth revising twice)
- Duplicate values break the `ROW_NUMBER` trick. De-duplicate first (`SELECT DISTINCT`).
- The trick needs the *same* ordering in both the number and the `ROW_NUMBER`. Dates are not integers: convert with `julianday()` or shift the date back by the row number.
- "Longer than 30 minutes" vs "30 minutes or more": decide the boundary on purpose, it changes the sessions.
- Merging intervals needs the running **maximum end** so far, not just the previous row's end (a long interval can swallow several short ones).
- Half-open intervals `[start, end)` make back-to-back bookings legal. Closed intervals need `<=`.
### When it matters
- Login streaks, uptime and downtime reports, missing invoice numbers, web analytics sessions, room/resource booking clashes, billing periods, change-history compression.
### One-line recap example
```sql
SELECT MIN(n), MAX(n) FROM (SELECT n, n - ROW_NUMBER() OVER (ORDER BY n) AS g FROM nums) GROUP BY g;  -- islands
```

## 01. Seed data
- `nums`: integers with holes. `login`: one row per user per active day. `ticket`: ids with some deleted.
- `ev`: click events with timestamps. `booking`: room bookings as `[start, end)` dates. `reading`: a device status log.
- Window function basics: `03-Window-Functions/01-Window-Functions`. Recursive CTEs: `02-Querying`.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE nums (n INTEGER PRIMARY KEY);
INSERT INTO nums VALUES (1),(2),(3),(7),(8),(10),(15),(16),(17),(18);

CREATE TABLE login (usr TEXT, day TEXT);
INSERT INTO login VALUES
 ('amy','2024-03-01'),('amy','2024-03-02'),('amy','2024-03-03'),('amy','2024-03-05'),
 ('amy','2024-03-06'),('amy','2024-03-09'),
 ('raj','2024-03-01'),('raj','2024-03-02'),('raj','2024-03-03'),('raj','2024-03-04'),
 ('raj','2024-03-05'),('raj','2024-03-08'),('raj','2024-03-09'),('raj','2024-03-09');

CREATE TABLE ticket (id INTEGER PRIMARY KEY, title TEXT);
INSERT INTO ticket VALUES (1,'a'),(2,'b'),(3,'c'),(5,'e'),(6,'f'),(9,'i'),(10,'j');

CREATE TABLE ev (usr TEXT, ts TEXT);
INSERT INTO ev VALUES
 ('u1','2024-05-01 09:00:00'),('u1','2024-05-01 09:10:00'),('u1','2024-05-01 09:25:00'),
 ('u1','2024-05-01 10:30:00'),('u1','2024-05-01 10:40:00'),('u1','2024-05-01 12:00:00'),
 ('u2','2024-05-01 09:05:00'),('u2','2024-05-01 09:50:00'),('u2','2024-05-01 10:20:00');

CREATE TABLE booking (id TEXT PRIMARY KEY, room TEXT, start TEXT, "end" TEXT);
INSERT INTO booking VALUES
 ('b1','R1','2024-06-01','2024-06-05'),('b2','R1','2024-06-04','2024-06-08'),
 ('b3','R1','2024-06-08','2024-06-10'),('b4','R1','2024-06-15','2024-06-17'),
 ('b5','R2','2024-06-02','2024-06-03'),('b6','R2','2024-06-03','2024-06-06');

CREATE TABLE reading (dev TEXT, t INTEGER, status TEXT, temp INTEGER);
INSERT INTO reading VALUES
 ('d1',1,'ok',20),('d1',2,'ok',22),('d1',3,'ok',25),('d1',4,'warn',31),('d1',5,'warn',35),
 ('d1',6,'ok',24),('d1',7,'down',NULL),('d1',8,'down',NULL),('d1',9,'ok',21),
 ('d2',1,'ok',19),('d2',2,'down',NULL),('d2',3,'down',NULL);
SELECT n FROM nums ORDER BY n;

n 
--
1 
2 
3 
7 
8 
10
15
16
17
18
(10 rows)



## 02. Gaps vs islands: the idea
- Look at `nums`: 1,2,3 | 7,8 | 10 | 15,16,17,18. Four islands: 1-3, 7-8, 10-10, 15-18.
- The gaps are what is between them: 4-6, 9, 11-14.
- Islands answer "what runs exist?". Gaps answer "what is missing?". Each can be derived from the other.
- We start with the classic trick for the islands. First, just see where the jumps are: a row whose step from the previous value is not 1 starts a new island.

In [3]:
%%sql
SELECT n, n - LAG(n) OVER (ORDER BY n) AS step FROM nums ORDER BY n;

n  | step
---+-----
1  | NULL
2  | 1   
3  | 1   
7  | 4   
8  | 1   
10 | 2   
15 | 5   
16 | 1   
17 | 1   
18 | 1   
(10 rows)



## 03. The ROW_NUMBER difference trick
- Number the rows 1,2,3... in order. Subtract from the value.
- Inside one island both go up by 1 each row, so the difference stays the same. At a gap the value jumps and the difference changes.
- Look at the `grp` column first, then group by it.

In [4]:
%%sql
SELECT n, ROW_NUMBER() OVER (ORDER BY n) AS rn,
       n - ROW_NUMBER() OVER (ORDER BY n) AS grp
FROM nums ORDER BY n;

n  | rn | grp
---+----+----
1  | 1  | 0  
2  | 2  | 0  
3  | 3  | 0  
7  | 4  | 3  
8  | 5  | 3  
10 | 6  | 4  
15 | 7  | 8  
16 | 8  | 8  
17 | 9  | 8  
18 | 10 | 8  
(10 rows)



In [5]:
%%sql
SELECT MIN(n) AS island_start, MAX(n) AS island_end, COUNT(*) AS size
FROM (SELECT n, n - ROW_NUMBER() OVER (ORDER BY n) AS grp FROM nums)
GROUP BY grp ORDER BY island_start;

island_start | island_end | size
-------------+------------+-----
1            | 3          | 3   
7            | 8          | 2   
10           | 10         | 1   
15           | 18         | 4   
(4 rows)



## 04. Gaps from the islands (LEAD)
- A gap runs from "end of one island + 1" to "start of next island - 1".
- Build the islands in a CTE, then use `LEAD` to look at the next island's start.
- The last island has no next one, so its `LEAD` is `NULL` and it drops out of the filter.

In [6]:
%%sql
WITH isl AS (
  SELECT MIN(n) AS s, MAX(n) AS e
  FROM (SELECT n, n - ROW_NUMBER() OVER (ORDER BY n) AS grp FROM nums) GROUP BY grp)
SELECT e + 1 AS gap_start, next_s - 1 AS gap_end, next_s - e - 1 AS missing
FROM (SELECT s, e, LEAD(s) OVER (ORDER BY s) AS next_s FROM isl)
WHERE next_s IS NOT NULL ORDER BY gap_start;

gap_start | gap_end | missing
----------+---------+--------
4         | 6       | 3      
9         | 9       | 1      
11        | 14      | 4      
(3 rows)



## 05. Consecutive date streaks
- Dates are not integers, so `date - ROW_NUMBER()` needs a conversion. Easiest in SQLite: shift the date back by `rn` days with `date(day, '-' || rn || ' days')`.
- Same idea: inside a streak the shifted date is constant.
- Do it per user with `PARTITION BY usr`.

In [7]:
%%sql
SELECT usr, day, ROW_NUMBER() OVER (PARTITION BY usr ORDER BY day) AS rn,
       date(day, '-' || ROW_NUMBER() OVER (PARTITION BY usr ORDER BY day) || ' days') AS grp
FROM (SELECT DISTINCT usr, day FROM login)
WHERE usr = 'amy' ORDER BY day;

usr | day        | rn | grp       
----+------------+----+-----------
amy | 2024-03-01 | 1  | 2024-02-29
amy | 2024-03-02 | 2  | 2024-02-29
amy | 2024-03-03 | 3  | 2024-02-29
amy | 2024-03-05 | 4  | 2024-03-01
amy | 2024-03-06 | 5  | 2024-03-01
amy | 2024-03-09 | 6  | 2024-03-03
(6 rows)



In [8]:
%%sql
WITH d AS (SELECT DISTINCT usr, day FROM login),
g AS (
  SELECT usr, day,
         date(day, '-' || ROW_NUMBER() OVER (PARTITION BY usr ORDER BY day) || ' days') AS grp
  FROM d)
SELECT usr, MIN(day) AS streak_start, MAX(day) AS streak_end, COUNT(*) AS days
FROM g GROUP BY usr, grp ORDER BY usr, streak_start;

usr | streak_start | streak_end | days
----+--------------+------------+-----
amy | 2024-03-01   | 2024-03-03 | 3   
amy | 2024-03-05   | 2024-03-06 | 2   
amy | 2024-03-09   | 2024-03-09 | 1   
raj | 2024-03-01   | 2024-03-05 | 5   
raj | 2024-03-08   | 2024-03-09 | 2   
(5 rows)



## 06. The duplicate-days gotcha
- Raj logged in twice on 2024-03-09. Without `DISTINCT` the row numbers advance but the date does not, so the trick breaks and one streak is split in two.
- Always collapse to one row per key first.

In [9]:
%%sql
-- Bad: duplicates inside the data. The 08-09 streak is wrongly cut into two pieces
WITH g AS (
  SELECT usr, day,
         date(day, '-' || ROW_NUMBER() OVER (PARTITION BY usr ORDER BY day) || ' days') AS grp
  FROM login WHERE usr = 'raj')
SELECT usr, MIN(day) AS streak_start, MAX(day) AS streak_end, COUNT(*) AS rows_counted
FROM g GROUP BY usr, grp ORDER BY streak_start;

usr | streak_start | streak_end | rows_counted
----+--------------+------------+-------------
raj | 2024-03-01   | 2024-03-05 | 5           
raj | 2024-03-08   | 2024-03-09 | 2           
raj | 2024-03-09   | 2024-03-09 | 1           
(3 rows)



In [10]:
%%sql
-- Good: DISTINCT first (section 05 does this): 2024-03-08 to 2024-03-09 is one streak of 2 days
WITH d AS (SELECT DISTINCT usr, day FROM login WHERE usr = 'raj'),
g AS (SELECT usr, day, date(day, '-' || ROW_NUMBER() OVER (ORDER BY day) || ' days') AS grp FROM d)
SELECT usr, MIN(day) AS streak_start, MAX(day) AS streak_end, COUNT(*) AS days
FROM g GROUP BY grp ORDER BY streak_start;

usr | streak_start | streak_end | days
----+--------------+------------+-----
raj | 2024-03-01   | 2024-03-05 | 5   
raj | 2024-03-08   | 2024-03-09 | 2   
(2 rows)



## 07. Longest streak per user
- Build streaks, then rank them per user by length. `ROW_NUMBER` with a tie-breaker (latest start) gives exactly one row per user.
- Use `RANK` instead if you want all tied streaks (see `04-Patterns/01-Top-N-And-Ranking-Patterns`).

In [11]:
%%sql
WITH d AS (SELECT DISTINCT usr, day FROM login),
g AS (SELECT usr, day, date(day, '-' || ROW_NUMBER() OVER (PARTITION BY usr ORDER BY day) || ' days') AS grp FROM d),
s AS (SELECT usr, MIN(day) AS streak_start, MAX(day) AS streak_end, COUNT(*) AS days FROM g GROUP BY usr, grp),
r AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY usr ORDER BY days DESC, streak_start DESC) AS rn FROM s)
SELECT usr, streak_start, streak_end, days FROM r WHERE rn = 1 ORDER BY usr;

usr | streak_start | streak_end | days
----+--------------+------------+-----
amy | 2024-03-01   | 2024-03-03 | 3   
raj | 2024-03-01   | 2024-03-05 | 5   
(2 rows)



## 08. Current streak as of a given day
- "Is the user still on a streak today?" A streak is current if it ends on the reference day (or the day before, if you allow today not to be over yet).
- We use a fixed reference date, `2024-03-09`, so the result never changes. In real code pass today's date as a parameter.

In [12]:
%%sql
WITH d AS (SELECT DISTINCT usr, day FROM login),
g AS (SELECT usr, day, date(day, '-' || ROW_NUMBER() OVER (PARTITION BY usr ORDER BY day) || ' days') AS grp FROM d),
s AS (SELECT usr, MIN(day) AS streak_start, MAX(day) AS streak_end, COUNT(*) AS days FROM g GROUP BY usr, grp)
SELECT usr, streak_start, days AS current_streak
FROM s WHERE streak_end >= date('2024-03-09', '-1 day') ORDER BY usr;

usr | streak_start | current_streak
----+--------------+---------------
amy | 2024-03-09   | 1             
raj | 2024-03-08   | 2             
(2 rows)



## 09. Missing IDs, way 1: compare each id to the next (LEAD)
- If the next id is more than `id + 1`, the ids in between are missing.
- No helper table needed, and it reports ranges (3 and 5 means 4 is missing).
- Cannot see missing ids **before the first** or **after the last** row (you need to know the expected range).

In [13]:
%%sql
SELECT id + 1 AS missing_from, next_id - 1 AS missing_to
FROM (SELECT id, LEAD(id) OVER (ORDER BY id) AS next_id FROM ticket)
WHERE next_id - id > 1 ORDER BY missing_from;

missing_from | missing_to
-------------+-----------
4            | 4         
7            | 8         
(2 rows)



## 10. Missing IDs, way 2: generate the full range and anti-join
- A recursive CTE produces every id from the expected min to max. Keep the ones that have no matching ticket.
- Use `NOT EXISTS` (not `NOT IN`, which breaks with `NULL`s, see `04-Patterns/01-Top-N-And-Ranking-Patterns`).
- Because you choose the range, you can also find ids missing before the first or after the last (here, the range runs to 12, so 11 and 12 show up even though the last ticket is 10).

In [14]:
%%sql
WITH RECURSIVE seq(id) AS (SELECT 1 UNION ALL SELECT id + 1 FROM seq WHERE id < 12)
SELECT id AS missing_id FROM seq s
WHERE NOT EXISTS (SELECT 1 FROM ticket t WHERE t.id = s.id) ORDER BY id;

missing_id
----------
4         
7         
8         
11        
12        
(5 rows)



## 11. Missing dates: build a calendar with a recursive CTE
- SQLite has no `generate_series` built in, so a recursive CTE produces a row per day.
- `date(d, '+1 day')` steps one day. The stop condition is the last day you want.
- Then `LEFT JOIN` the facts onto the calendar and look for `NULL`: those are the missing dates.

In [15]:
%%sql
WITH RECURSIVE cal(d) AS (
  SELECT '2024-03-01' UNION ALL SELECT date(d, '+1 day') FROM cal WHERE d < '2024-03-10')
SELECT cal.d AS missing_day
FROM cal LEFT JOIN (SELECT DISTINCT day FROM login WHERE usr = 'amy') a ON a.day = cal.d
WHERE a.day IS NULL ORDER BY cal.d;

missing_day
-----------
2024-03-04 
2024-03-07 
2024-03-08 
2024-03-10 
(4 rows)



## 12. Zero-filling a report with the calendar
- A normal `GROUP BY day` silently skips days with no rows, so charts get holes.
- Put the calendar on the left of a `LEFT JOIN`, count the right side, and `COALESCE`/`COUNT(col)` gives 0 for empty days.
- Count a column of the right table (`COUNT(l.usr)`), not `COUNT(*)`, or empty days show 1.

In [16]:
%%sql
WITH RECURSIVE cal(d) AS (
  SELECT '2024-03-01' UNION ALL SELECT date(d, '+1 day') FROM cal WHERE d < '2024-03-09')
SELECT cal.d AS day, COUNT(l.usr) AS logins
FROM cal LEFT JOIN (SELECT DISTINCT usr, day FROM login) l ON l.day = cal.d
GROUP BY cal.d ORDER BY cal.d;

day        | logins
-----------+-------
2024-03-01 | 2     
2024-03-02 | 2     
2024-03-03 | 2     
2024-03-04 | 1     
2024-03-05 | 2     
2024-03-06 | 1     
2024-03-07 | 0     
2024-03-08 | 1     
2024-03-09 | 2     
(9 rows)



## 13. Sessionization with LAG + running SUM
- Rule: a new session starts when the gap since the user's previous event is **more than 30 minutes** (1800 seconds).
- Step 1: `LAG(ts)` gives the previous event time. Step 2: flag = 1 if first event or gap > 1800 else 0. Step 3: running `SUM(flag)` = session number.
- Time differences in SQLite: `strftime('%s', ts)` gives epoch seconds, subtract them.

In [17]:
%%sql
WITH x AS (
  SELECT usr, ts,
         strftime('%s', ts) - strftime('%s', LAG(ts) OVER (PARTITION BY usr ORDER BY ts)) AS gap_sec
  FROM ev)
SELECT usr, ts, gap_sec,
       CASE WHEN gap_sec IS NULL OR gap_sec > 1800 THEN 1 ELSE 0 END AS new_session,
       SUM(CASE WHEN gap_sec IS NULL OR gap_sec > 1800 THEN 1 ELSE 0 END)
           OVER (PARTITION BY usr ORDER BY ts) AS session_no
FROM x ORDER BY usr, ts;

usr | ts                  | gap_sec | new_session | session_no
----+---------------------+---------+-------------+-----------
u1  | 2024-05-01 09:00:00 | NULL    | 1           | 1         
u1  | 2024-05-01 09:10:00 | 600     | 0           | 1         
u1  | 2024-05-01 09:25:00 | 900     | 0           | 1         
u1  | 2024-05-01 10:30:00 | 3900    | 1           | 2         
u1  | 2024-05-01 10:40:00 | 600     | 0           | 2         
u1  | 2024-05-01 12:00:00 | 4800    | 1           | 3         
u2  | 2024-05-01 09:05:00 | NULL    | 1           | 1         
u2  | 2024-05-01 09:50:00 | 2700    | 1           | 2         
u2  | 2024-05-01 10:20:00 | 1800    | 0           | 2         
(9 rows)



## 14. Session summary
- Group by user and session number for the count of events, start, end and length.
- The boundary matters: u2's events at 09:50 and 10:20 are exactly 30 minutes apart (1800 s). With `> 1800` they stay in one session. With `>= 1800` they would split.

In [18]:
%%sql
WITH x AS (
  SELECT usr, ts,
         strftime('%s', ts) - strftime('%s', LAG(ts) OVER (PARTITION BY usr ORDER BY ts)) AS gap_sec
  FROM ev),
s AS (
  SELECT usr, ts,
         SUM(CASE WHEN gap_sec IS NULL OR gap_sec > 1800 THEN 1 ELSE 0 END) OVER (PARTITION BY usr ORDER BY ts) AS session_no
  FROM x)
SELECT usr, session_no, COUNT(*) AS events, MIN(ts) AS started, MAX(ts) AS ended,
       (strftime('%s', MAX(ts)) - strftime('%s', MIN(ts))) / 60 AS minutes
FROM s GROUP BY usr, session_no ORDER BY usr, session_no;

usr | session_no | events | started             | ended               | minutes
----+------------+--------+---------------------+---------------------+--------
u1  | 1          | 3      | 2024-05-01 09:00:00 | 2024-05-01 09:25:00 | 25     
u1  | 2          | 2      | 2024-05-01 10:30:00 | 2024-05-01 10:40:00 | 10     
u1  | 3          | 1      | 2024-05-01 12:00:00 | 2024-05-01 12:00:00 | 0      
u2  | 1          | 1      | 2024-05-01 09:05:00 | 2024-05-01 09:05:00 | 0      
u2  | 2          | 2      | 2024-05-01 09:50:00 | 2024-05-01 10:20:00 | 30     
(5 rows)



## 15. Overlapping intervals: detect clashes
- Two intervals overlap when each starts before the other ends: `a.start < b.end AND b.start < a.end`.
- Self-join with `a.id < b.id` so each pair appears once and a booking is not compared with itself.
- With half-open `[start, end)`, b2 -> b3 (ends 06-08, starts 06-08) is **not** a clash. `<=` would wrongly report it.

In [19]:
%%sql
-- Good
SELECT a.id AS first, b.id AS second, a.room
FROM booking a JOIN booking b ON a.room = b.room AND a.id < b.id
 AND a.start < b."end" AND b.start < a."end"
ORDER BY a.room, a.id, b.id;

first | second | room
------+--------+-----
b1    | b2     | R1  
(1 row)



In [20]:
%%sql
-- Bad: <= treats back-to-back bookings as clashes (extra rows: b2/b3 and b5/b6)
SELECT a.id AS first, b.id AS second, a.room
FROM booking a JOIN booking b ON a.room = b.room AND a.id < b.id
 AND a.start <= b."end" AND b.start <= a."end"
ORDER BY a.room, a.id, b.id;

first | second | room
------+--------+-----
b1    | b2     | R1  
b2    | b3     | R1  
b5    | b6     | R2  
(3 rows)



## 16. Merge overlapping intervals
- Goal: collapse bookings into continuous "occupied" blocks per room.
- Step 1: running max of `end` over all **earlier** rows (`ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING`).
- Step 2: a row starts a new block if it begins after that running max (or has no earlier rows). Step 3: running `SUM` of flags numbers the blocks.
- Here touching intervals (start equals previous end) merge into one block. Use `>=` for the flag if touching should stay separate.
- Using only the previous row's end (`LAG`) is wrong: one long interval can cover several short ones.

In [21]:
%%sql
WITH o AS (
  SELECT room, start, "end",
         MAX("end") OVER (PARTITION BY room ORDER BY start, "end"
                          ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS prev_max
  FROM booking),
f AS (SELECT *, CASE WHEN prev_max IS NULL OR start > prev_max THEN 1 ELSE 0 END AS new_block FROM o),
g AS (SELECT *, SUM(new_block) OVER (PARTITION BY room ORDER BY start, "end") AS blk FROM f)
SELECT room, MIN(start) AS block_start, MAX("end") AS block_end
FROM g GROUP BY room, blk ORDER BY room, block_start;

room | block_start | block_end 
-----+-------------+-----------
R1   | 2024-06-01  | 2024-06-10
R1   | 2024-06-15  | 2024-06-17
R2   | 2024-06-02  | 2024-06-06
(3 rows)



## 17. Interval coverage and free gaps
- Save the merged blocks as a view. From it: total days booked (no double counting of overlaps) and the free time between blocks.
- Naively summing each booking's length double counts the b1/b2 overlap. Compare the two numbers.

In [22]:
%%sql
CREATE VIEW merged AS
WITH o AS (
  SELECT room, start, "end",
         MAX("end") OVER (PARTITION BY room ORDER BY start, "end"
                          ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS prev_max
  FROM booking),
f AS (SELECT *, CASE WHEN prev_max IS NULL OR start > prev_max THEN 1 ELSE 0 END AS new_block FROM o),
g AS (SELECT *, SUM(new_block) OVER (PARTITION BY room ORDER BY start, "end") AS blk FROM f)
SELECT room, MIN(start) AS block_start, MAX("end") AS block_end FROM g GROUP BY room, blk;

SELECT m.room,
       CAST(SUM(julianday(block_end) - julianday(block_start)) AS INTEGER) AS days_covered,
       (SELECT CAST(SUM(julianday("end") - julianday(start)) AS INTEGER) FROM booking b WHERE b.room = m.room) AS days_if_summed
FROM merged m GROUP BY m.room ORDER BY m.room;

room | days_covered | days_if_summed
-----+--------------+---------------
R1   | 11           | 12            
R2   | 4            | 4             
(2 rows)



In [23]:
%%sql
-- free gaps between blocks per room
SELECT room, block_end AS free_from, next_start AS free_until,
       CAST(julianday(next_start) - julianday(block_end) AS INTEGER) AS free_days
FROM (SELECT room, block_start, block_end,
             LEAD(block_start) OVER (PARTITION BY room ORDER BY block_start) AS next_start FROM merged)
WHERE next_start IS NOT NULL ORDER BY room, free_from;

room | free_from  | free_until | free_days
-----+------------+------------+----------
R1   | 2024-06-10 | 2024-06-15 | 5        
(1 row)



## 18. Status-change runs
- A device log says ok, ok, ok, warn, warn, ok, down... Each uninterrupted stretch of the same status is a run.
- Method A (flag + running sum): flag a row when the status differs from the previous one. Works for anything.
- Method B (two row numbers): `ROW_NUMBER() over time` minus `ROW_NUMBER() over (time, within status)` is constant inside a run.
- Note `ok` forms three separate runs for d1 (readings 1-3, 6 and 9), because other statuses sit between them.

In [24]:
%%sql
-- Method A
WITH f AS (
  SELECT dev, t, status,
         CASE WHEN status IS LAG(status) OVER (PARTITION BY dev ORDER BY t) THEN 0 ELSE 1 END AS changed
  FROM reading),
g AS (SELECT *, SUM(changed) OVER (PARTITION BY dev ORDER BY t) AS run_no FROM f)
SELECT dev, run_no, status, MIN(t) AS from_t, MAX(t) AS to_t, COUNT(*) AS readings
FROM g GROUP BY dev, run_no ORDER BY dev, run_no;

dev | run_no | status | from_t | to_t | readings
----+--------+--------+--------+------+---------
d1  | 1      | ok     | 1      | 3    | 3       
d1  | 2      | warn   | 4      | 5    | 2       
d1  | 3      | ok     | 6      | 6    | 1       
d1  | 4      | down   | 7      | 8    | 2       
d1  | 5      | ok     | 9      | 9    | 1       
d2  | 1      | ok     | 1      | 1    | 1       
d2  | 2      | down   | 2      | 3    | 2       
(7 rows)



In [25]:
%%sql
-- Method B gives the same runs (grp is a label, not a run counter)
WITH g AS (
  SELECT dev, t, status,
         ROW_NUMBER() OVER (PARTITION BY dev ORDER BY t)
       - ROW_NUMBER() OVER (PARTITION BY dev, status ORDER BY t) AS grp
  FROM reading)
SELECT dev, status, MIN(t) AS from_t, MAX(t) AS to_t, COUNT(*) AS readings
FROM g GROUP BY dev, status, grp ORDER BY dev, from_t;

dev | status | from_t | to_t | readings
----+--------+--------+------+---------
d1  | ok     | 1      | 3    | 3       
d1  | warn   | 4      | 5    | 2       
d1  | ok     | 6      | 6    | 1       
d1  | down   | 7      | 8    | 2       
d1  | ok     | 9      | 9    | 1       
d2  | ok     | 1      | 1    | 1       
d2  | down   | 2      | 3    | 2       
(7 rows)



## 19. First and last row of each run
- Want the temperature at the start and end of each run, not just the time range?
- Number rows inside each run twice (ascending and descending) and pick rank 1 of each, or use `FIRST_VALUE`/`LAST_VALUE` with a full frame.
- Here the run is identified by method A's `run_no`. `down` runs have `NULL` temps, which is shown as is.

In [26]:
%%sql
WITH f AS (
  SELECT dev, t, status, temp,
         CASE WHEN status IS LAG(status) OVER (PARTITION BY dev ORDER BY t) THEN 0 ELSE 1 END AS changed
  FROM reading),
g AS (SELECT *, SUM(changed) OVER (PARTITION BY dev ORDER BY t) AS run_no FROM f)
SELECT DISTINCT dev, run_no, status,
       FIRST_VALUE(temp) OVER w AS temp_first,
       LAST_VALUE(temp)  OVER w AS temp_last
FROM g
WINDOW w AS (PARTITION BY dev, run_no ORDER BY t ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
ORDER BY dev, run_no;

dev | run_no | status | temp_first | temp_last
----+--------+--------+------------+----------
d1  | 1      | ok     | 20         | 25       
d1  | 2      | warn   | 31         | 35       
d1  | 3      | ok     | 24         | 24       
d1  | 4      | down   | NULL       | NULL     
d1  | 5      | ok     | 21         | 21       
d2  | 1      | ok     | 19         | 19       
d2  | 2      | down   | NULL       | NULL     
(7 rows)



## 20. Run-length encoding of a string
- RLE turns `aaabccdddd` into `a3 b1 c2 d4`. It is the same island problem: runs of equal characters.
- Split the string into characters with a recursive CTE, then reuse method B from section 18.
- Finally `GROUP_CONCAT` the pieces in order (the `ORDER BY` inside `GROUP_CONCAT` needs SQLite 3.44+; without it the order is not guaranteed).

In [27]:
%%sql
WITH RECURSIVE ch(i, c) AS (
  SELECT 1, substr('aaabccdddd', 1, 1)
  UNION ALL SELECT i + 1, substr('aaabccdddd', i + 1, 1) FROM ch WHERE i < length('aaabccdddd')),
g AS (
  SELECT i, c, ROW_NUMBER() OVER (ORDER BY i) - ROW_NUMBER() OVER (PARTITION BY c ORDER BY i) AS grp FROM ch),
runs AS (SELECT c, grp, MIN(i) AS pos, COUNT(*) AS n FROM g GROUP BY c, grp)
SELECT c, n, pos FROM runs ORDER BY pos;

c | n | pos
--+---+----
a | 3 | 1  
b | 1 | 4  
c | 2 | 5  
d | 4 | 7  
(4 rows)



In [28]:
%%sql
WITH RECURSIVE ch(i, c) AS (
  SELECT 1, substr('aaabccdddd', 1, 1)
  UNION ALL SELECT i + 1, substr('aaabccdddd', i + 1, 1) FROM ch WHERE i < length('aaabccdddd')),
g AS (
  SELECT i, c, ROW_NUMBER() OVER (ORDER BY i) - ROW_NUMBER() OVER (PARTITION BY c ORDER BY i) AS grp FROM ch),
runs AS (SELECT c, grp, MIN(i) AS pos, COUNT(*) AS n FROM g GROUP BY c, grp)
SELECT GROUP_CONCAT(c || n, ' ' ORDER BY pos) AS encoded FROM runs;

encoded    
-----------
a3 b1 c2 d4
(1 row)



## 21. Putting it together: a streak report with gaps
- For each user: number of streaks, longest streak, and the longest break between streaks.
- Streaks come from the date trick, breaks from `LEAD` over the streaks. All in one query, still readable because each CTE does one job.
- Amy: streaks 03-01..03-03, 03-05..03-06, 03-09 so breaks of 1 and 2 missed days.

In [29]:
%%sql
WITH d AS (SELECT DISTINCT usr, day FROM login),
g AS (SELECT usr, day, date(day, '-' || ROW_NUMBER() OVER (PARTITION BY usr ORDER BY day) || ' days') AS grp FROM d),
s AS (SELECT usr, MIN(day) AS s_start, MAX(day) AS s_end, COUNT(*) AS days FROM g GROUP BY usr, grp),
b AS (SELECT usr, days,
             CAST(julianday(LEAD(s_start) OVER (PARTITION BY usr ORDER BY s_start)) - julianday(s_end) AS INTEGER) - 1 AS missed
      FROM s)
SELECT usr, COUNT(*) AS streaks, MAX(days) AS longest_streak, MAX(missed) AS longest_break_days
FROM b GROUP BY usr ORDER BY usr;

usr | streaks | longest_streak | longest_break_days
----+---------+----------------+-------------------
amy | 3       | 3              | 2                 
raj | 2       | 5              | 2                 
(2 rows)

